# IPCA, principal component regression and PLS benchmarks

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


In [ ]:
%%writefile utils.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
__author__ = 'Hans Andrias Gregoriussen'

import numpy as np

np.random.seed(42)


def format_time(t):
    """Return a formatted time string 'HH:MM:SS
    based on a numeric time() value"""
    m, s = divmod(t, 60)
    h, m = divmod(m, 60)
    return f'{h:0>2.0f}:{m:0>2.0f}:{s:0>2.0f}'

class MultipleTimeSeriesCV:
    """Generates tuples of train_idx and test_idx pairs
    Assumes the MultiIndex contains levels 'symbol' and 'date'
    purges overlapping outcomes"""

    def __init__(self,
                 n_splits=3,
                 train_period_length=126,
                 test_period_length=21,
                 lookahead=None,
                 date_idx='date',
                 shuffle=False):
        self.n_splits = n_splits
        self.lookahead = lookahead
        if self.lookahead is None:
            self.lookahead = 0
        self.test_length = test_period_length
        self.train_length = train_period_length
        self.shuffle = shuffle
        self.date_idx = date_idx





    def split(self, X, y=None, groups=None):
        unique_dates = X.index.get_level_values(self.date_idx).unique()
        days = sorted(unique_dates, reverse=True)
        split_idx = []
        for i in range(self.n_splits):
            test_end_idx = i * self.test_length
            test_start_idx = test_end_idx + self.test_length
            train_end_idx = test_start_idx + self.lookahead
            train_start_idx = train_end_idx + self.train_length + self.lookahead
            split_idx.append([train_start_idx, train_end_idx,
                              test_start_idx, test_end_idx])

        dates = X.reset_index()[[self.date_idx]]
        for train_start, train_end, test_start, test_end in split_idx:

            train_idx = dates[(dates[self.date_idx] > days[train_start])
                              & (dates[self.date_idx] <= days[train_end])].index
            test_idx = dates[(dates[self.date_idx] > days[test_start])
                             & (dates[self.date_idx] <= days[test_end])].index
            if self.shuffle:
                np.random.shuffle(list(train_idx))
            yield train_idx.to_numpy(),test_idx.to_numpy()

    def get_n_splits(self, X, y, groups=None):
        return self.n_splits


**CV SCHEME WITH VALIDATION SPLIT**

In [ ]:
%%writefile utils.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
__author__ = 'Hans Andrias Gregoriussen'

import numpy as np

np.random.seed(42)


def format_time(t):
    """Return a formatted time string 'HH:MM:SS
    based on a numeric time() value"""
    m, s = divmod(t, 60)
    h, m = divmod(m, 60)
    return f'{h:0>2.0f}:{m:0>2.0f}:{s:0>2.0f}'

class MultipleTimeSeriesCV:
    """Generates tuples of train_idx, val_idx and test_idx pairs
    Assumes the MultiIndex contains levels 'symbol' and 'date'
    purges overlapping outcomes"""

    def __init__(self,
                 n_splits=3,
                 train_period_length=126,
                 test_period_length=21,
                 val_period_length=21,
                 lookahead=None,
                 date_idx='date',
                 shuffle=False):
        self.n_splits = n_splits
        self.lookahead = lookahead
        if self.lookahead is None:
            self.lookahead = 0
        self.test_length = test_period_length
        self.val_length = val_period_length
        self.train_length = train_period_length
        self.shuffle = shuffle
        self.date_idx = date_idx





    def split(self, X, y=None, groups=None):
        unique_dates = X.index.get_level_values(self.date_idx).unique()
        days = sorted(unique_dates, reverse=True)
        split_idx = []
        for i in range(self.n_splits):
            test_end_idx = i * self.test_length
            test_start_idx = test_end_idx + self.test_length
            val_end_idx = test_start_idx + self.lookahead
            val_start_idx = val_end_idx + self.val_length + self.lookahead
            train_end_idx = val_start_idx + self.lookahead
            train_start_idx = train_end_idx + self.train_length + self.lookahead
            split_idx.append([train_start_idx, train_end_idx,
                              val_start_idx, val_end_idx,
                              test_start_idx, test_end_idx])

        dates = X.reset_index()[[self.date_idx]]
        for train_start, train_end, val_start, val_end, test_start, test_end in split_idx:

            train_idx = dates[(dates[self.date_idx] > days[train_start])
                              & (dates[self.date_idx] <= days[train_end])].index
            val_idx = dates[(dates[self.date_idx] > days[val_start])
                              & (dates[self.date_idx] <= days[val_end])].index
            test_idx = dates[(dates[self.date_idx] > days[test_start])
                             & (dates[self.date_idx] <= days[test_end])].index
            if self.shuffle:
                np.random.shuffle(list(train_idx))
            yield train_idx.to_numpy(),val_idx.to_numpy(), test_idx.to_numpy()

    def get_n_splits(self, X, y, groups=None):
        return self.n_splits



class DateTimeSeriesCV:
    """Generates tuples of train_idx,val_idx and test_idx pairs
    Assumes the index is of datetime type
    purges overlapping outcomes"""

    def __init__(self, n_splits=3, train_period_length=126, val_period_length=21, test_period_length=21, lookahead=None, shuffle=False):
        self.n_splits = n_splits
        self.lookahead = lookahead
        if self.lookahead is None:
            self.lookahead = 0
        self.test_length = test_period_length
        self.val_length = val_period_length
        self.train_length = train_period_length
        self.shuffle = shuffle

    def split(self, X, y=None, groups=None):
        unique_dates = X.index.unique()
        days = sorted(unique_dates, reverse=True)

        split_idx = []
        for i in range(self.n_splits):
            test_end_idx = i * self.test_length
            test_start_idx = test_end_idx + self.test_length
            val_end_idx = test_start_idx + self.lookahead
            val_start_idx = val_end_idx + self.val_length + self.lookahead
            train_end_idx = val_start_idx + self.lookahead
            train_start_idx = train_end_idx + self.train_length + self.lookahead

            if train_start_idx >= len(days):
                break  # Stop if we've gone beyond the dataset

            split_idx.append([train_start_idx, train_end_idx, val_start_idx, val_end_idx, test_start_idx, test_end_idx])

        for train_start, train_end, val_start, val_end, test_start, test_end in split_idx:
            train_idx = X.index[(X.index > days[train_start]) & (X.index <= days[train_end])]
            val_idx = X.index[(X.index > days[val_start]) & (X.index <= days[val_end])]
            test_idx = X.index[(X.index > days[test_start]) & (X.index <= days[test_end])]

            if self.shuffle:
                train_idx = np.random.permutation(train_idx)

            yield train_idx, val_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits


In [ ]:
from utils import format_time, MultipleTimeSeriesCV


In [ ]:
# from google.colab import drive
# import pandas as pd
# drive.mount('/content/drive')

# file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta_input_CRSP.parquet'

# # Make sure merged_df1 is a DataFrame and not a function or any other type.
# merged_df_padded = pd.read_parquet(file_path)
# print(merged_df_padded)

# import pandas as pd

# # Assuming merged_df_padded is your dataframe with columns 'date' and 'permno'

# # Step 1: Set the 'date' and 'permno' columns as the MultiIndex
# merged_df_padded.set_index(['date', 'permno'], inplace=True)

# # Step 2: Extract the "date" level from the MultiIndex and convert to a DatetimeIndex
# date_level = merged_df_padded.index.get_level_values('date')
# date_level_with_freq = pd.DatetimeIndex(date_level).to_period('M').to_timestamp()

# # Check the new date level
# print("Date level with frequency set (sample):", date_level_with_freq[:10])
# print("New date level frequency:", date_level_with_freq.freq)

# # Step 3: Create a new MultiIndex with the updated date level
# new_index = pd.MultiIndex.from_arrays(
#     [date_level_with_freq, merged_df_padded.index.get_level_values('permno')],
#     names=merged_df_padded.index.names
# )

# # Reindex the dataframe with this new MultiIndex
# merged_df_padded.index = new_index

# # Set the frequency explicitly on the 'date' level in the MultiIndex
# merged_df_padded.index.levels[0].freq = 'MS'

# # Validate the new index frequency
# print("New index frequency:", merged_df_padded.index.levels[0].freq)  # Should print <MonthBegin>

# # Check the unique dates to ensure they have the frequency set
# print("Unique dates (sample):", merged_df_padded.index.levels[0][:10])

# # Rename the column in the dataframe
# merged_df_padded.rename(columns={'market_weighted_portfolio': 'characteristic_market_weighted_portfolio'}, inplace=True)

# # Print the dataframe to confirm the change
# print(merged_df_padded)


# # Group by 'date' level and sample 200 'permno' for each date
# merged_df_padded = merged_df_padded.groupby(level='date').apply(lambda x: x.sample(n=200, random_state=1))

# # Remove the additional index created by groupby apply
# merged_df_padded.index = merged_df_padded.index.droplevel(0)

# # Check the result
# print(merged_df_padded)


# **LOAD STOCK CHARACTERISTICS MONTHLY DATA**




In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install pandas pyarrow
import pandas as pd

# Define the file path
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

# Load the Parquet file into a DataFrame
df = pd.read_parquet(file_path, engine='pyarrow')

# Check the loaded data
print(df.head())


In [ ]:
import pandas as pd

# Reset the index to convert 'month' to a column
df_reset = df.reset_index()

# Ensure the 'month' column is in datetime format
df_reset['month'] = pd.to_datetime(df_reset['month'])

# Set the index back to 'month' and 'permno'
df = df_reset.set_index(['month', 'permno'])

# Sort the MultiIndex to ensure it is lexsorted
df = df.sort_index()

# Define the date range
start_date = '1960-01-01'
end_date = '2020-01-01'

# Filter the DataFrame for the specified date range
df = df.loc[start_date:end_date]

# Display the filtered DataFrame
print(df.index)


In [ ]:
import pandas as pd
from sklearn.preprocessing import quantile_transform
from tqdm import tqdm

# Assuming 'df' is your dataframe that has been filtered and processed so far.

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]

# Function to carry forward annual and quarterly data within a MultiIndex
def carry_forward(data, frequency, index_name):
    if frequency == 'annual':
        period = 12
    elif frequency == 'quarterly':
        period = 3
    else:
        raise ValueError("Frequency should be 'annual' or 'quarterly'")

    carried_data = data.copy()
    for i in range(1, period):
        carried_data = carried_data.groupby(level=index_name).ffill()
    return carried_data

# Apply carry forward for annual and quarterly data with progress bars
for code in tqdm(annual_codes, desc="Processing annual codes"):
    char_column = f'characteristic_{code}'
    df[char_column] = carry_forward(df[char_column], 'annual', 'permno')

for code in tqdm(quarterly_codes, desc="Processing quarterly codes"):
    char_column = f'characteristic_{code}'
    df[char_column] = carry_forward(df[char_column], 'quarterly', 'permno')

# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
narratives = [f'characteristic_{code}' for code in all_codes]

# Enable the tqdm pandas extension
tqdm.pandas()

# Rank normalization function
def rank_normalize(group, columns):
    for col in columns:
        values = group[col].values.reshape(-1, 1)
        if len(values) > 1:  # Ensure there is more than one value to normalize
            transformed_values = quantile_transform(values, copy=True, n_quantiles=min(len(values), 100)).flatten()
            group[col] = transformed_values * 2 - 1
        else:
            group[col] = 0  # Handle case with a single value to avoid all -1.0
    return group

# Reset index to avoid duplicate levels after groupby and progress_apply
df_reset = df.reset_index()

# Apply rank normalization by month with progress bar
df_reset = df_reset.groupby('month').progress_apply(lambda group: rank_normalize(group, columns=narratives))

# Reset index to flatten the MultiIndex generated by groupby-apply
df_reset.reset_index(drop=True, inplace=True)

# Set the correct index
df1 = df_reset.set_index(['month', 'permno'])

# Display the dataframe head to check the result
print(df1.head())


In [ ]:
import pandas as pd

# Ensure the dataframe is sorted by 'permno' and 'month'
df1 = df1.sort_index()

# Shift the 'ret_excess' column to calculate forward returns for the next month
df1['forward_ret_excess_1m'] = df1.groupby('permno')['ret_excess'].shift(-1)

# # If you want to calculate forward returns for the next 3 months
# df1['forward_ret_excess_3m'] = df1.groupby('permno')['ret_excess'].shift(-3)

# # If you want to calculate forward returns for the next 6 months
# df1['forward_ret_excess_6m'] = df1.groupby('permno')['ret_excess'].shift(-6)

# # Display the dataframe head to check the result
# print(df1.head())

# # Optionally, drop rows where forward returns are NaN (at the end of each permno series)
# df1.dropna(subset=['forward_ret_excess_1m', 'forward_ret_excess_3m', 'forward_ret_excess_6m'], how='all', inplace=True)

# Display the dataframe head after dropping NaNs
print(df1.head())


In [ ]:
# import pandas as pd

# # Assuming df1 is your DataFrame

# # Step 1: Calculate the total market capitalization for each month
# df1['total_mktcap'] = df1.groupby('month')['mktcap_lag'].transform('sum')

# # Step 2: Calculate the weight of each stock's market cap relative to the total market cap for that month
# df1['weight'] = df1['mktcap_lag'] / df1['total_mktcap']

# # Step 3: Calculate the market-weighted portfolio return for each stock
# # This involves multiplying the excess return by the weight
# df1['market_weighted_return'] = df1['ret_excess'] * df1['weight']

# # Step 4: Aggregate to get the market-weighted portfolio return for each month
# # Summing these weighted returns will give the market-weighted portfolio return for the month
# df1['market_weighted_portfolio'] = df1.groupby('month')['market_weighted_return'].transform('sum')

# # Display the updated DataFrame to check the results
# print(df1.head())


In [ ]:
import pandas as pd

# Assuming df1 is your DataFrame with a MultiIndex of ['month', 'permno']

# Drop rows with any missing values
df1_clean = df1.dropna()

# Reset index to make 'month' and 'permno' columns for easier grouping
df1_clean_reset = df1_clean.reset_index()

# Group by 'month' and count unique 'permno' entries for each month
permno_counts = df1_clean_reset.groupby('month')['permno'].nunique()

# Calculate the average number of 'permno' entries across all months
average_permno_count = permno_counts.mean()

# Display the average number of 'permno' entries per month
print(f"Average number of permnos per month: {average_permno_count:.2f}")

# Optionally, display the counts for each month
print("Number of permnos per month:")
print(permno_counts.mean)


In [ ]:
# Optionally, display the counts for each month
print("Number of permnos per month:")
print(permno_counts.mean())

In [ ]:
# Rename the index level from 'month' to 'date'
df1_clean.index.rename(['date', 'permno'], inplace=True)


In [ ]:
# # Rename the column 'market_weighted_portfolio' to 'characteristic_market_weighted_portfolio'
# df1_clean.rename(columns={'market_weighted_portfolio': 'characteristic_market_weighted_portfolio'}, inplace=True)


In [ ]:
# Swap the index levels
df1_clean = df1_clean.swaplevel('date', 'permno').sort_index()



In [ ]:
x_t_all_months = df1_clean

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]
#,
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristic_columns = [f'characteristic_{code}' for code in all_codes]

# Add "ret_excess" to the list of columns to select
columns_to_select = characteristic_columns + ['ret_excess']
#forward_ret_excess_1m
# Select only the specified columns from merged_df_padded
x_t_all_months = df1_clean[columns_to_select]

n_characeristics = len(characteristic_columns)
n_tickers = x_t_all_months.index.get_level_values('permno').unique()
print(n_characeristics)
print(n_tickers)

# Print the selected DataFrame to verify
print(x_t_all_months.head())


In [ ]:
!pip install ipca

# **IPCA characteristic importance TOTAL $R^2$**

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from utils import format_time, MultipleTimeSeriesCV
from ipca import InstrumentedPCA
from sklearn.metrics import r2_score
from pathlib import Path
from tqdm import tqdm

# Seed for reproducibility
np.random.seed(42)

# Constants
YEAR = 12

# Initialize the time series cross-validator
cv_portfolio = MultipleTimeSeriesCV(n_splits=20, train_period_length=22*YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)

# Split the data
cv_dataset_splits = list(cv_portfolio.split(x_t_all_months))

# Function to preprocess the train and test data
def get_train_test_data(x_t_all_months, train_idx, test_idx):
    train_set = x_t_all_months.iloc[train_idx]
    test_set = x_t_all_months.iloc[test_idx]

    y_train = train_set['ret_excess'].to_numpy()
    x_train = train_set.drop(columns=['ret_excess'])

    y_test = test_set['ret_excess'].to_numpy()
    x_test = test_set.drop(columns=['ret_excess'])

    # Remove rows with all zero columns in x_train and x_test
    train_all_zeros = (x_train == 0).all(axis=1)
    x_train_filtered = x_train.loc[~train_all_zeros]
    y_train_filtered = y_train[~train_all_zeros]

    test_all_zeros = (x_test == 0).all(axis=1)
    x_test_filtered = x_test.loc[~test_all_zeros]
    y_test_filtered = y_test[~test_all_zeros]

    return x_train_filtered, y_train_filtered, x_test_filtered, y_test_filtered

# Path to save the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_IPCA')
hdf5_path = results_path / 'predictions.h5'

# Ensure the results directory exists
if not results_path.exists():
    results_path.mkdir(parents=True)

# Create the HDF5 file initially in 'table' format if it doesn't exist
if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()

# Function to shuffle a single characteristic
def shuffle_characteristic(df, column):
    df_shuffled = df.copy()
    df_shuffled[column] = np.random.permutation(df_shuffled[column].values)
    return df_shuffled

# Function to calculate R^2 score
def calculate_r2(x_train, y_train, x_test, y_test, n_factors):
    regr = InstrumentedPCA(n_factors=n_factors, intercept=False, max_iter=500, iter_tol=1e-05, alpha=0.0, l1_ratio=1.0, n_jobs=-1, backend='loky')
    regr.fit(X=x_train, y=y_train)
    y_pred = regr.predict(X=x_test, mean_factor=True)
    y_pred = np.array(y_pred).flatten()
    return r2_score(y_test, y_pred)

num_factors = [5]

# Dictionary to store feature importance for each factor
all_feature_importances = {}

# Loop over each factor and calculate average R^2 for all folds
for n_factors in tqdm(num_factors):  # Adjust the range as needed
    r2_scores = []
    feature_importance = {}

    for fold, (train_idx, test_idx) in enumerate(cv_dataset_splits):
        x_train, y_train, x_test, y_test = get_train_test_data(x_t_all_months, train_idx, test_idx)

        # Baseline R^2 score
        baseline_r2 = calculate_r2(x_train, y_train, x_test, y_test, n_factors)
        r2_scores.append(baseline_r2)

        # Feature importance by shuffling
        for column in x_train.columns:
            x_train_shuffled = shuffle_characteristic(x_train, column)
            shuffled_r2 = calculate_r2(x_train_shuffled, y_train, x_test, y_test, n_factors)
            if column not in feature_importance:
                feature_importance[column] = []
            feature_importance[column].append(baseline_r2 - shuffled_r2)

        # Save the actual and predicted returns
        prediction_df = pd.DataFrame({
            'date': x_test.index.get_level_values('date'),
            'permno': x_test.index.get_level_values('permno').astype(str),
            'predicted returns': y_pred,
            'actual returns': y_test
        }).reset_index(drop=True)

        print(f'| FACTOR : {n_factors} |FOLD: {fold}|TEST R^2 :{baseline_r2}|')
        with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
            store.put(f'fwd_predictions/n_factors_{n_factors}_fold_{fold}', prediction_df, format='table')

        print(f"Fold {fold+1}, Factors {n_factors} - Baseline R-squared (R²): {baseline_r2}")

    # Calculate and print the average R^2 score for the current number of factors
    average_r2 = np.mean(r2_scores)
    print(f"Average R-squared (R²) for {n_factors} factors across all folds: {average_r2}")

    # Calculate and normalize feature importance
    avg_feature_importance = {k: np.mean(v) for k, v in feature_importance.items()}
    total_importance = sum(avg_feature_importance.values())
    normalized_feature_importance = {k: v / total_importance for k, v in avg_feature_importance.items()}
    all_feature_importances[n_factors] = normalized_feature_importance

    # Print feature importance
    sorted_importance = sorted(normalized_feature_importance.items(), key=lambda item: item[1], reverse=True)
    print(f"Feature importance for {n_factors} factors:")
    for feature, importance in sorted_importance:
        print(f"{feature}: {importance:.4f}")

    # Create a bar plot for each factor
    plt.figure(figsize=(10, 6))
    sns.barplot(x=list(normalized_feature_importance.values()), y=list(normalized_feature_importance.keys()))
    plt.xlabel('Importance')
    plt.ylabel('Features')
    plt.title(f'Feature Importance for {n_factors} Factors')
    plt.show()

# Create a DataFrame for the heatmap
heatmap_df = pd.DataFrame(all_feature_importances).fillna(0)

# Remove the prefix "characteristic_" from the index names if needed
heatmap_df.index = heatmap_df.index.str.replace('characteristic_', '')

# Plot the heatmap
plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Factors')
plt.ylabel('Variables')
plt.title('Variable Importances for 5 Factors')
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from utils import format_time, MultipleTimeSeriesCV
from ipca import InstrumentedPCA
from sklearn.metrics import r2_score
import time
from pathlib import Path
from tqdm import tqdm
# Seed for reproducibility
np.random.seed(42)

# Constants
YEAR = 12

# Initialize the time series cross-validator
cv_portfolio = MultipleTimeSeriesCV(n_splits=20, train_period_length=22* YEAR, test_period_length=1* YEAR, lookahead=0 * YEAR)

# Split the data
cv_dataset_splits = list(cv_portfolio.split(x_t_all_months))

# Function to preprocess the train and test data
def get_train_test_data(x_t_all_months, train_idx, test_idx):
    train_set = x_t_all_months.iloc[train_idx]
    test_set = x_t_all_months.iloc[test_idx]

    y_train = train_set['ret_excess'].to_numpy()
    x_train = train_set.drop(columns=['ret_excess'])

    y_test = test_set['ret_excess'].to_numpy()
    x_test = test_set.drop(columns=['ret_excess'])

    # Remove rows with all zero columns in x_train and x_test
    train_all_zeros = (x_train == 0).all(axis=1)
    x_train_filtered = x_train.loc[~train_all_zeros]
    y_train_filtered = y_train[~train_all_zeros]

    test_all_zeros = (x_test == 0).all(axis=1)
    x_test_filtered = x_test.loc[~test_all_zeros]
    y_test_filtered = y_test[~test_all_zeros]

    return x_train_filtered, y_train_filtered, x_test_filtered, y_test_filtered

# Path to save the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_IPCA')
hdf5_path = results_path / 'predictions.h5'

# Ensure the results directory exists
if not results_path.exists():
    results_path.mkdir(parents=True)

# Create the HDF5 file initially in 'table' format if it doesn't exist
if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()

# Loop over each factor and calculate average R^2 for all folds
for n_factors in tqdm(range(1, 7)):  # Adjust the range as needed
    r2_scores = []
    for fold, (train_idx, test_idx) in enumerate(cv_dataset_splits):
        x_train, y_train, x_test, y_test = get_train_test_data(x_t_all_months, train_idx, test_idx)

        # Initialize the IPCA model
        regr = InstrumentedPCA(n_factors=n_factors, intercept=False, max_iter=500, iter_tol=1e-05, alpha=0.0, l1_ratio=1.0, n_jobs=-1, backend='loky')

        # Fit the model on the training data
        regr.fit(X=x_train, y=y_train)

        # Generate out-of-sample predictions
        try:
            y_pred = regr.predict(X=x_test,mean_factor=True)
            #, y=y_test, mean_factor=False
            # Ensure y_pred is a numpy array for consistency
            y_pred = np.array(y_pred).flatten()

            # Compute R^2 score
            r2 = r2_score(y_test, y_pred)
            r2_scores.append(r2)

            # Save the actual and predicted returns
            prediction_df = pd.DataFrame({
                'date': x_test.index.get_level_values('date'),
                'permno': x_test.index.get_level_values('permno').astype(str),
                'predicted returns': y_pred,
                'actual returns': y_test
            }).reset_index(drop=True)

            print(f'| FACTOR : {n_factors} |FOLD: {fold}|TEST R^2 :{r2}|')
            with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
                store.put(f'fwd_predictions/n_factors_{n_factors}_fold_{fold}', prediction_df, format='table')

            print(f"Fold {fold+1}, Factors {n_factors} - R-squared (R²): {r2}")
        except ValueError as e:
            print(f"Fold {fold+1}, Factors {n_factors} - Error during prediction: {e}")

    # Calculate and print the average R^2 score for the current number of factors
    average_r2 = np.mean(r2_scores)
    print(f"Average R-squared (R²) for {n_factors} factors across all folds: {average_r2}")


# **LOAD PREDICTIONS FOR FWD AND TOTAL**

In [ ]:
import pandas as pd
from pathlib import Path
!pip install tables

# Ensure Google Drive is mounted
from google.colab import drive
drive.mount('/content/drive')


# Define the paths to the HDF5 files
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_IPCA1')
fwd_predictions_path = results_path / 'fwd_predictions.h5'
predictions_path = results_path / 'predictions.h5'

# Function to load and display the contents of an HDF5 file
def load_hdf5_file(hdf5_path):
    if not hdf5_path.exists():
        print(f"File {hdf5_path} does not exist.")
        return

    with pd.HDFStore(hdf5_path, mode='r') as store:
        keys = store.keys()
        print(f"Contents of {hdf5_path}:")
        for key in keys:
            df = store.get(key)
            print(f"\nKey: {key}")
            print(df.head())

# Load and display the contents of fwd_predictions.h5 if it exists
if fwd_predictions_path.exists():
    load_hdf5_file(fwd_predictions_path)
else:
    print(f"{fwd_predictions_path} does not exist.")

# Load and display the contents of predictions.h5 if it exists
if predictions_path.exists():
    load_hdf5_file(predictions_path)
else:
    print(f"{predictions_path} does not exist.")


In [ ]:
import pandas as pd
from pathlib import Path

# Ensure Google Drive is mounted
from google.colab import drive
drive.mount('/content/drive')


# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/IPCA_SIMULATIONS_NON_LINEAR_TOTAL')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# List of factors to load
factors = [1]  # Add other factor values if needed

# Dictionary to store the loaded DataFrames
scores_dict = {}

# Loop through each factor and load the corresponding DataFrame
for n_factors in factors:
    hdf5_path = results_path / f'scores_{n_factors}.h5'
    scores_df = pd.read_hdf(str(hdf5_path), key='scores')
    scores_dict[n_factors] = scores_df
    print(f"Scores for {n_factors} factors:")
    print(scores_df)
    print("\n")

# Access individual DataFrames from the dictionary
# For example, to access the DataFrame for 1 factor:
scores_1_factor = scores_dict[1]
print(scores_1_factor)


In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import r2_score
import numpy as np

# Load the predictions
def load_predictions(hdf5_path, n_factors_range, n_folds):
    predictions_list = []

    # Open the HDF5 file
    with pd.HDFStore(hdf5_path, mode='r') as store:
        # Loop over each factor and fold to read the data
        for n_factors in n_factors_range:
            for fold in range(n_folds):
                key = f'predictions/n_factors_{n_factors}_fold_{fold}'
                if key in store:
                    df = store.get(key)
                    df['n_factors'] = n_factors
                    df['fold'] = fold
                    predictions_list.append(df)
                else:
                    print(f"Key {key} not found in the HDF5 file.")

    # Concatenate all the DataFrames into one
    combined_df = pd.concat(predictions_list, axis=0, ignore_index=True)

    return combined_df

# Define the range of factors and the number of folds
n_factors_range = range(1, 7)  # Adjust the range as needed
n_folds = 20  # Replace with the actual number of folds used

# Path to the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_IPCA')
hdf5_path = results_path / 'predictions.h5'

# Load the predictions
all_predictions_df = load_predictions(hdf5_path, n_factors_range, n_folds)

# Load the stock characteristics
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'
df = pd.read_parquet(file_path, engine='pyarrow')

# Preprocess the stock characteristics DataFrame
df = df.rename(columns={'month': 'date'})
df = df.set_index(['date', 'permno'])
df = df.sort_index()

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Select the characteristics columns
characteristics_data = df[characteristics]

# Fill missing values with the median for each group (date)
def fill_missing_with_median(group):
    return group.fillna(group.median())

characteristics_data = characteristics_data.groupby(level='date').apply(fill_missing_with_median)
characteristics_data.index = characteristics_data.index.droplevel(0)

# Ensure 'date' columns are of datetime type
all_predictions_df['date'] = pd.to_datetime(all_predictions_df['date'])
characteristics_data.index = characteristics_data.index.set_levels(pd.to_datetime(characteristics_data.index.levels[0]), level=0)

# Ensure 'permno' columns are of the same type
all_predictions_df['permno'] = all_predictions_df['permno'].astype(str).str.split('.').str[0].astype(int)
characteristics_data.index = characteristics_data.index.set_levels(characteristics_data.index.levels[1].astype(int), level=1)

# Inspect the DataFrames
print("Sample rows from all_predictions_df:")
print(all_predictions_df.head())

print("Sample rows from characteristics_data:")
print(characteristics_data.head())

# Perform an inner join between the predictions and the characteristics
merged_df = all_predictions_df.set_index(['date', 'permno']).join(characteristics_data, how='inner')

# Check the merged DataFrame
print(f"Merged DataFrame has {len(merged_df)} rows.")
print(merged_df.head())

# Function to calculate x_t for a given month using Ridge Regression
def calculate_xt(df, date, returns_column, alpha=0):
    # Filter DataFrame for the given date
    df_month = df.loc[date]

    # Separate returns and characteristics
    r_t = df_month[returns_column].values
    characteristic_columns = df_month.drop(columns=[returns_column]).columns
    Z_t_minus_1 = df_month.drop(columns=[returns_column]).values

    # Calculate (Z_{t-1}'Z_{t-1})^{-1}Z_{t-1}r_t using Ridge Regression
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])

    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)

    # Calculate x_t
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t

    return x_t, characteristic_columns

# Calculate x_t for all unique dates in the dataset
def calculate_xt_all_dates(df, returns_column, alpha=1e-6):
    xt_results = {}
    characteristic_columns = None

    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t, characteristic_columns = calculate_xt(df, date, returns_column, alpha)
        xt_results[date] = x_t

    # Convert dictionary to DataFrame
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')

    # Assign original characteristic names as column names
    if characteristic_columns is not None:
        xt_df.columns = characteristic_columns

    # Preserve the index name
    xt_df.index.name = 'date'

    return xt_df

# Calculate x_t for predicted returns for each n_factors
xt_pred_dict = {}
for n_factors in n_factors_range:
    df_filtered = merged_df[merged_df['n_factors'] == n_factors]
    xt_pred_dict[n_factors] = calculate_xt_all_dates(df_filtered, 'predicted returns')

# Calculate x_t for actual returns for each n_factors
xt_true_dict = {}
for n_factors in n_factors_range:
    df_filtered = merged_df[merged_df['n_factors'] == n_factors]
    xt_true_dict[n_factors] = calculate_xt_all_dates(df_filtered, 'actual returns')

# Display the x_t DataFrames for one example factor
print("x_t for predicted returns with 1 factor:")
print(xt_pred_dict[1].head())

print("x_t for actual returns with 1 factor:")
print(xt_true_dict[1].head())

# Function to calculate R^2
def calculate_r2(xt_true, xt_pred):
    y_true = xt_true.values.flatten()
    y_pred = xt_pred.values.flatten()

    # Calculate residuals
    residuals = y_true - y_pred

    # Calculate the sum of squares of residuals
    ssr = np.sum(np.square(residuals))

    # Calculate the total sum of squares
    tss = np.sum(np.square(y_true - np.mean(y_true)))

    # Calculate R^2
    r_squared = 1 - (ssr / tss)

    return r_squared

# Calculate R^2 for each n_factors
r2_scores = {}
for n_factors in n_factors_range:
    xt_true = xt_true_dict[n_factors]
    xt_pred = xt_pred_dict[n_factors]
    r2_scores[n_factors] = calculate_r2(xt_true, xt_pred)

# Print the R^2 scores
print("R^2 scores for each n_factors:")
for n_factors, r2 in r2_scores.items():
    print(f"n_factors={n_factors}: R^2={r2}")


# **PCA**

**COMMENT: for forward returns, change "ret_excess" with "forward_ret_excess_1m" and "predictions.h5" in the results path with "fwd_predictions.h5"**

In [ ]:
import pandas as pd
import numpy as np
from utils import format_time, MultipleTimeSeriesCV
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
import time
from pathlib import Path

# Seed for reproducibility
np.random.seed(42)

# Constants
YEAR = 12

# Initialize the time series cross-validator
cv_portfolio = MultipleTimeSeriesCV(n_splits=20, train_period_length=22 * YEAR, val_period_length=0* YEAR, test_period_length=1 * YEAR, lookahead=0 * YEAR)

# Split the data
cv_dataset_splits = list(cv_portfolio.split(x_t_all_months))

# Function to preprocess the train and test data
def get_train_test_data(x_t_all_months, train_idx, val_idx, test_idx):
    train_set = x_t_all_months.iloc[train_idx]
    test_set = x_t_all_months.iloc[test_idx]
#forward_ret_excess_1m
    y_train = train_set['ret_excess'].to_numpy()

    x_train = train_set.drop(columns=['ret_excess'])

    y_test = test_set['ret_excess'].to_numpy()
    x_test = test_set.drop(columns=['ret_excess'])


    # Remove rows with all zero columns in x_train and x_test
    train_all_zeros = (x_train == 0).all(axis=1)
    x_train_filtered = x_train.loc[~train_all_zeros]
    y_train_filtered = y_train[~train_all_zeros]

    test_all_zeros = (x_test == 0).all(axis=1)
    x_test_filtered = x_test.loc[~test_all_zeros]
    y_test_filtered = y_test[~test_all_zeros]

    return x_train_filtered, y_train_filtered, x_test_filtered, y_test_filtered

# Path to save the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_PCA')
hdf5_path = results_path / 'predictions.h5'
#fwd_predictions.h5
# Ensure the results directory exists
if not results_path.exists():
    results_path.mkdir(parents=True)

# Create the HDF5 file initially in 'table' format if it doesn't exist
if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()
factors = [1,2,3,4,5,6]


# Loop over each factor and calculate average R^2 for all folds
for n_factors in tqdm(factors):  # Adjust the range as needed
    r2_scores = []
    for fold, (train_idx,val_idx, test_idx) in enumerate(cv_dataset_splits):
        x_train, y_train, x_test, y_test = get_train_test_data(x_t_all_months, train_idx, val_idx, test_idx)

        # Initialize the PCA model
        pca = PCA(n_components=n_factors)

        # Fit the PCA model on the training data to get the latent factors
        x_train_pca = pca.fit_transform(x_train)

        # Fit a regression model on the PCA-transformed data (latent factors)
        from sklearn.linear_model import LinearRegression
        regr = LinearRegression()
        regr.fit(x_train_pca, y_train)

        # Transform the test data using the same PCA
        x_test_pca = pca.transform(x_test)

        # Generate out-of-sample predictions
        y_pred = regr.predict(x_test_pca)

        # Ensure y_pred is a numpy array for consistency
        y_pred = np.array(y_pred).flatten()

        # Compute R^2 score
        r2 = r2_score(y_test, y_pred)
        r2_scores.append(r2)

        # Save the actual and predicted returns
        prediction_df = pd.DataFrame({
            'date': x_test.index.get_level_values('date'),
            'permno': x_test.index.get_level_values('permno').astype(str),
            'y_pred': y_pred,
            'y_true': y_test
        }).reset_index(drop=True)

        with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
            store.put(f'predictions/n_factors_{n_factors}_fold_{fold}', prediction_df, format='table')

        print(f"Fold {fold+1}, Factors {n_factors} - R-squared (R²): {r2}")

    # Calculate and print the average R^2 score for the current number of factors
    average_r2 = np.mean(r2_scores)
    print(f"Average R-squared (R²) for {n_factors} factors across all folds: {average_r2}")


# **PLS**

In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.cross_decomposition import PLSRegression
# from sklearn.metrics import r2_score
# from sklearn.preprocessing import StandardScaler
# from tqdm import tqdm
# from pathlib import Path
# from utils import format_time, MultipleTimeSeriesCV  # Assuming you have this utility function

# # Seed for reproducibility
# np.random.seed(42)

# # Constants
# YEAR = 12

# # Initialize the time series cross-validator
# cv_portfolio = MultipleTimeSeriesCV(n_splits=20, train_period_length=22 * YEAR, val_period_length=0 * YEAR, test_period_length=1 * YEAR, lookahead=0 * YEAR)

# # Split the data
# cv_dataset_splits = list(cv_portfolio.split(x_t_all_months))

# # Function to preprocess the train and test data
# def get_train_test_data(x_t_all_months, train_idx, val_idx, test_idx):
#     train_set = x_t_all_months.iloc[train_idx]
#     test_set = x_t_all_months.iloc[test_idx]

#     y_train = train_set['forward_ret_excess_1m'].to_numpy()
#     x_train = train_set.drop(columns=['forward_ret_excess_1m'])

#     y_test = test_set['forward_ret_excess_1m'].to_numpy()
#     x_test = test_set.drop(columns=['forward_ret_excess_1m'])
# #forward_ret_excess_1m
#     # Remove rows with all zero columns in x_train and x_test
#     train_all_zeros = (x_train == 0).all(axis=1)
#     x_train_filtered = x_train.loc[~train_all_zeros]
#     y_train_filtered = y_train[~train_all_zeros]

#     test_all_zeros = (x_test == 0).all(axis=1)
#     x_test_filtered = x_test.loc[~test_all_zeros]
#     y_test_filtered = y_test[~test_all_zeros]

#     return x_train_filtered, y_train_filtered, x_test_filtered, y_test_filtered

# # Path to save the HDF5 file
# results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/predictive_results_PLS')
# hdf5_path = results_path / 'predictions.h5'

# # Ensure the results directory exists
# if not results_path.exists():
#     results_path.mkdir(parents=True)

# # Create the HDF5 file initially in 'table' format if it doesn't exist
# if not hdf5_path.exists():
#     with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
#         store.close()

# # Factors to consider for PLS
# factors = [1, 2, 3, 4, 5, 6]

# # Loop over each factor and calculate average R^2 for all folds for PLS
# for n_factors in tqdm(factors):
#     r2_scores_pls = []
#     all_predictions_pls = []  # Initialize all_predictions_pls here
#     for fold, (train_idx, val_idx, test_idx) in enumerate(cv_dataset_splits):
#         x_train, y_train, x_test, y_test = get_train_test_data(x_t_all_months, train_idx, val_idx, test_idx)

#         # Standardize the predictors
#         scaler = StandardScaler()
#         x_train_scaled = scaler.fit_transform(x_train)
#         x_test_scaled = scaler.transform(x_test)

#         # Apply PLS
#         pls = PLSRegression(n_components=n_factors)
#         pls.fit(x_train_scaled, y_train)

#         # Predict and evaluate for PLS
#         y_pred_pls = pls.predict(x_test_scaled)
#         r2_pls = r2_score(y_test, y_pred_pls)

#         # Create a DataFrame for the current fold's predictions
#         fold_predictions = pd.DataFrame({
#             'fold': fold,
#             'actual': y_test,
#             'predicted': y_pred_pls.flatten()
#         })
#         print(f'Average R^2 for PLS with {n_factors}| fold : {fold}| factors: {r2_pls}')


#        # Add the fold predictions to the list
#         all_predictions_pls.append(fold_predictions)

#         # Save R² scores
#         r2_scores_pls.append(r2_pls)

#     # Combine all fold predictions into a single DataFrame for this factor
#     combined_predictions = pd.concat(all_predictions_pls, ignore_index=True)

#     # Save combined predictions to HDF5 file
#     with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
#         store.put(f'/pls_predictions/factors_{n_factors}', combined_predictions)

#     avg_r2_pls = np.mean(r2_scores_pls)
#     print(f'Average R^2 for PLS with {n_factors} factors: {avg_r2_pls}')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from pathlib import Path
from utils import format_time, MultipleTimeSeriesCV  # Assuming you have this utility function

# Seed for reproducibility
np.random.seed(42)

# Constants
YEAR = 12

# Initialize the time series cross-validator
cv_portfolio = MultipleTimeSeriesCV(n_splits=20, train_period_length=22 * YEAR, val_period_length=0 * YEAR, test_period_length=1 * YEAR, lookahead=0 * YEAR)

# Split the data
cv_dataset_splits = list(cv_portfolio.split(x_t_all_months))

# Function to preprocess the train and test data
def get_train_test_data(x_t_all_months, train_idx, val_idx, test_idx):
    train_set = x_t_all_months.iloc[train_idx]
    test_set = x_t_all_months.iloc[test_idx]

    y_train = train_set['ret_excess'].to_numpy()
    x_train = train_set.drop(columns=['ret_excess'])

    y_test = test_set['ret_excess'].to_numpy()
    x_test = test_set.drop(columns=['ret_excess'])

    # Extract date and permno from the multi-index and reset index
    test_dates = test_set.index.get_level_values('date')
    test_permnos = test_set.index.get_level_values('permno')

    # Remove rows with all zero columns in x_train and x_test
    train_all_zeros = (x_train == 0).all(axis=1)
    x_train_filtered = x_train.loc[~train_all_zeros]
    y_train_filtered = y_train[~train_all_zeros]

    test_all_zeros = (x_test == 0).all(axis=1)
    x_test_filtered = x_test.loc[~test_all_zeros]
    y_test_filtered = y_test[~test_all_zeros]

    # Also filter the dates and permnos accordingly
    test_dates_filtered = test_dates[~test_all_zeros]
    test_permnos_filtered = test_permnos[~test_all_zeros]

    return x_train_filtered, y_train_filtered, x_test_filtered, y_test_filtered, test_dates_filtered, test_permnos_filtered

# Path to save the HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/total_results_PLS')
hdf5_path = results_path / 'predictions.h5'

# Ensure the results directory exists
if not results_path.exists():
    results_path.mkdir(parents=True)

# Create the HDF5 file initially in 'table' format if it doesn't exist
if not hdf5_path.exists():
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()

# Factors to consider for PLS
factors = [1, 2, 3, 4, 5, 6]

# Loop over each factor and calculate average R^2 for all folds for PLS
for n_factors in tqdm(factors):
    r2_scores_pls = []
    all_predictions_pls = []  # Initialize all_predictions_pls here
    for fold, (train_idx, val_idx, test_idx) in enumerate(cv_dataset_splits):
        x_train, y_train, x_test, y_test, test_dates, test_permnos = get_train_test_data(x_t_all_months, train_idx, val_idx, test_idx)

        # Standardize the predictors
        scaler = StandardScaler()
        x_train_scaled = scaler.fit_transform(x_train)
        x_test_scaled = scaler.transform(x_test)

        # Apply PLS
        pls = PLSRegression(n_components=n_factors)
        pls.fit(x_train_scaled, y_train)

        # Predict and evaluate for PLS
        y_pred_pls = pls.predict(x_test_scaled)
        r2_pls = r2_score(y_test, y_pred_pls)

        # Create a DataFrame for the current fold's predictions
        fold_predictions = pd.DataFrame({
            'fold': fold,
            'date': test_dates,
            'permno': test_permnos,
            'actual': y_test,
            'predicted': y_pred_pls.flatten()
        })

        print(f'Average R^2 for PLS with {n_factors}| fold : {fold}| factors: {r2_pls}')

        # Add the fold predictions to the list
        all_predictions_pls.append(fold_predictions)

        # Save R² scores
        r2_scores_pls.append(r2_pls)

    # Combine all fold predictions into a single DataFrame for this factor
    combined_predictions = pd.concat(all_predictions_pls, ignore_index=True)

    # Save combined predictions to HDF5 file
    with pd.HDFStore(hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.put(f'/pls_predictions/factors_{n_factors}', combined_predictions)

    avg_r2_pls = np.mean(r2_scores_pls)
    print(f'Average R^2 for PLS with {n_factors} factors: {avg_r2_pls}')


In [ ]:
fold_predictions

**Load predictions**

In [ ]:
import pandas as pd
from pathlib import Path

# Path to the saved HDF5 file
results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/total_results_PLS')
hdf5_path = results_path / 'predictions.h5'

# Function to load predictions from HDF5 for a specific factor
def load_saved_predictions(hdf5_path, factor):
    with pd.HDFStore(hdf5_path, mode='r') as store:
        key = f'/pls_predictions/factors_{factor}'
        if key in store:
            predictions = store[key]
            return predictions
        else:
            raise ValueError(f"No predictions found for {factor} factors in the HDF5 file.")

# Example of loading predictions for a specific number of factors
factor_to_load = 4  # Specify the number of factors you want to load predictions for
try:
    saved_predictions = load_saved_predictions(hdf5_path, factor_to_load)
    print(f'Successfully loaded predictions for {factor_to_load} factors.')
    print(saved_predictions.head())  # Display the first few rows of the predictions DataFrame
except ValueError as e:
    print(e)
